# Lesson 01 - AI, Machine Learning, and Deep Learning

**What this notebook does:** We build two tiny versions of the same task - deciding if a support ticket is urgent - so you can *see* the difference between the old way (a human writes rules) and the new way (the program learns from examples). No libraries needed yet, just plain Python.

## Part 1: The old way - a human writes the rules

Here is the traditional way to solve "is this support ticket urgent?": a person sits down and writes a list of keywords that mean "urgent." If any keyword shows up in the ticket, we call it urgent. This is 100% hand-written logic - the computer is not learning anything, it is just checking a list we gave it.

In [1]:
def rule_based_urgency(ticket_text):
    text = ticket_text.lower()
    urgent_keywords = ["urgent", "immediately", "asap", "broken", "not working", "refund now"]
    for keyword in urgent_keywords:
        if keyword in text:
            return "urgent"
    return "not urgent"

In [2]:
test_tickets = [
    "This is urgent, my order never arrived!",
    "The item I received is broken and I need this fixed immediately",
    "Just wondering when my package will ship, no rush",
    "I really need someone to look at this right now, it can not wait",
]

for ticket in test_tickets:
    print(ticket, "->", rule_based_urgency(ticket))

This is urgent, my order never arrived! -> urgent
The item I received is broken and I need this fixed immediately -> urgent
Just wondering when my package will ship, no rush -> not urgent
I really need someone to look at this right now, it can not wait -> not urgent


## Where the rulebook breaks

Look at the last ticket in the list above: "I really need someone to look at this right now, it can not wait." A human reading that would instantly call it urgent. But our rulebook only checks for exact words like "urgent" or "immediately," and this ticket does not use any of them, so the rule-based function gets it wrong. It will confidently say "not urgent."

This is the core weakness of hand-written rules: the real world has infinite ways to phrase the same idea, and a human cannot possibly write a keyword for every single one.

## Part 2: The new way - learning from examples

Instead of a human guessing which keywords matter, we will show the program a small set of already-labeled tickets (text plus the correct answer) and let it work out, purely from counting, which words tend to show up in urgent tickets versus calm ones. Nobody tells it "urgent" is an important word - it discovers that on its own by comparing the two piles.

In [ ]:
training_examples = [
    ("My order still has not arrived and I need it right now", "urgent"),
    ("This is extremely time sensitive please help right now", "urgent"),
    ("The app crashed and I cannot log in please fix this immediately", "urgent"),
    ("Just wondering when my package will ship no rush", "not urgent"),
    ("Can you tell me more about your return policy sometime", "not urgent"),
    ("I have a general question about sizing whenever you get a chance", "not urgent"),
]

## What "learning" means here

We are going to split every training sentence into individual words, then keep two tally sheets: one counting how often each word appears across all the *urgent* examples, and one counting how often each word appears across all the *not urgent* examples. A word like "right" or "now" will pile up on the urgent tally sheet just because it happens to appear in urgent examples more often. Nobody is telling the program that - it falls out naturally from counting.

Then, to score a brand new ticket, we add up how "urgent-leaning" versus "calm-leaning" its words are, based on those two tally sheets.

In [ ]:
from collections import Counter

def tokenize(text):
    return text.lower().replace(",", "").replace(".", "").split()

urgent_word_counts = Counter()
not_urgent_word_counts = Counter()

for text, label in training_examples:
    words = tokenize(text)
    if label == "urgent":
        urgent_word_counts.update(words)
    else:
        not_urgent_word_counts.update(words)

In [ ]:
def learned_urgency_score(ticket_text):
    words = tokenize(ticket_text)
    score = 0
    for word in words:
        score += urgent_word_counts.get(word, 0)
        score -= not_urgent_word_counts.get(word, 0)
    return score

def learned_urgency(ticket_text):
    score = learned_urgency_score(ticket_text)
    return "urgent" if score > 0 else "not urgent"

## The real test

Now let's try the *exact same ticket that fooled our rulebook* - "I really need someone to look at this right now, it can not wait" - plus one brand new calm ticket that was never in the training examples. Neither sentence exists in `training_examples`, so if the learned approach gets them right, it is genuinely generalizing, not just memorizing.

In [ ]:
new_tickets = [
    "I really need someone to look at this right now, it can not wait",
    "Can you tell me about your shipping policy whenever you get a chance",
]

for ticket in new_tickets:
    print(ticket)
    print("  rule-based says:", rule_based_urgency(ticket))
    print("  learned approach says:", learned_urgency(ticket), "(score:", learned_urgency_score(ticket), ")")

## Why this matters

The learned approach correctly flags the first ticket as urgent even though it never saw that exact sentence before, and it correctly leaves the second one as not urgent. It did this by noticing that words like "right," "now," and "need" leaned urgent in the examples we gave it, and combining that evidence - not by matching a fixed keyword list.

This tiny word-counting trick is a simplified preview of a real technique called Naive Bayes, which you do not need to remember by name yet. The important idea is the shift itself: **we stopped writing rules and started learning from labeled examples.** That shift is what the word "machine learning" means, and every technique in Phase 2 of this course is a more powerful, more careful version of exactly what you just watched happen above.